
### CREATE FLAG PARAMETER

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text('incremental_flag', '0')

#It is in string type
#So we have to use '0' or '1' when we are using it 

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')


### CREATING DIMENSION MODEL

In [0]:
%sql

SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`


### Gold - Dim_Dealer

In [0]:
df_src = spark.sql('''
                     SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`
                     ''')
display(df_src)

In [0]:
df_dealer = spark.sql('''
                     SELECT DISTINCT(Dealer_ID) AS Dealer_ID, DealerName FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`
                     ''')
display(df_dealer)

In [0]:
#If Table not exists - just bring the schema

if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    df_sink = spark.sql('''
    SELECT dim_dealer_key, Dealer_ID, DealerName FROM cars_cata.gold.dim_dealer
    ''')
else:
    df_sink = spark.sql('''
    SELECT 1 as dim_dealer_key, Dealer_ID, DealerName FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales` WHERE 1=0
    ''')
 

In [0]:
display(df_sink)

In [0]:
# Filtering new records and old records

df_filter = df_dealer.join(df_sink, df_dealer.Dealer_ID == df_sink.Dealer_ID, 'left').select(df_dealer['Dealer_ID'], df_dealer['DealerName'], df_sink['dim_dealer_key'])




In [0]:
df_filter_old = df_filter.filter(col("dim_dealer_key").isNotNull())
df_filter_new = df_filter.filter(col("dim_dealer_key").isNull()).select("Dealer_ID", "DealerName")

In [0]:
display(df_filter_new)


#### Create Surrogate Key


###### Fetch the Max Surrogate Key 

In [0]:
#Initial run - start from 1
#Incremental run - start from max(key)+1 

if incremental_flag == '0':
    max_value = 0
else:
    max_value_df = spark.sql(''' SELECT max(dim_dealer_key) FROM   cars_cata.gold.dim_dealer ''')
    max_value = max_value_df.collect()[0][0]


###### Create Surrogate Key

In [0]:
df_filter_new = df_filter_new.withColumn('dim_dealer_key', max_value+monotonically_increasing_id()+1)

In [0]:
display(df_filter_new)



#### Create final df

In [0]:
df_final = df_filter_new.union(df_filter_old)
display(df_final)


### SCD TYPE 1 - UPSERT

In [0]:
#if - incremental run else for initial run

if spark.catalog.tableExists('cars_cata.gold.dim_dealer'):
    delta_tbl = DeltaTable.forPath(spark,'abfss://gold@adlscarproject.dfs.core.windows.net/dim_dealer' )
    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.dim_dealer_key = src.dim_dealer_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.format("delta")\
        .mode('overwrite')\
        .option('path', 'abfss://gold@adlscarproject.dfs.core.windows.net/dim_dealer')\
        .saveAsTable('cars_cata.gold.dim_dealer')

In [0]:
%sql
SELECT * FROM cars_cata.gold.dim_dealer